# EchoAI — Qwen2.5-3B-Instruct fine-tune · v2 (single-day, lighter config)

All fixes from the v1 attempt baked in:
- torchao upgrade for peft
- Clean bf16 model load (no `device_map='auto'` double-allocation)
- LoRA r=16 (was r=64)
- MAX_LEN=384 (was 1024)
- No gradient checkpointing (we have T4 memory)
- 2 epochs (was 3)
- save_steps=150 (~7 min cadence; disconnect-safe)
- transformers 5.x API: `processing_class` not `tokenizer`
- GGUF Q4_K_M conversion at the end for deployment

**Expected runtime:** ~75-90 min training + 10 min eval + 10 min GGUF.

## Cell 1 — GPU + Drive

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'No GPU! Runtime → Change runtime type → T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/qwen_ft'
DRIVE_DIR = '/content/drive/MyDrive/echoai_qwen_ft'
os.makedirs(WORK, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Workspace :', WORK)
print('Drive dir :', DRIVE_DIR)

## Cell 2 — Install (one-shot, then restart prompt)

In [ ]:
!pip install -q -U "torchao>=0.16.0" "peft>=0.14.0" "accelerate>=1.2.0" "datasets>=3.1.0"
print('\nInstalled. → Now click Runtime → Restart session, then continue from Cell 3.')

### ⚠️ After running Cell 2, click `Runtime → Restart session`, then come back and run Cell 3 onward.

Drive is still mounted after restart. Pip installs persist on disk.

## Cell 3 — Re-mount Drive (after restart) + check versions

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/qwen_ft'
DRIVE_DIR = '/content/drive/MyDrive/echoai_qwen_ft'
os.makedirs(WORK, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

import transformers, peft, torchao, accelerate
print('transformers:', transformers.__version__)
print('peft        :', peft.__version__)
print('torchao     :', torchao.__version__, '(must be >= 0.16)')
print('accelerate  :', accelerate.__version__)

## Cell 4 — Get the 4 dataset files

**Choose ONE of the two options below.**

**Option A** — if datasets are already in this account's Drive at `MyDrive/echoai_qwen_ft/`.

In [ ]:
# OPTION A — copy from Drive
import shutil
needed = ['dataset_v1_train.jsonl','dataset_v1_val.jsonl','val_prompts.jsonl','heldout_prompts.jsonl']
missing = []
for f in needed:
    src = f'{DRIVE_DIR}/{f}'
    dst = f'{WORK}/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        n = sum(1 for _ in open(dst))
        print(f'  {f}: {n} lines')
    else:
        missing.append(f)
if missing:
    print('\nMISSING from Drive:', missing, '— run Option B below.')
else:
    print('\n[✓] All 4 datasets staged. Skip Option B.')

**Option B** — upload from local laptop (run this if Option A reported missing files).

In [ ]:
# OPTION B — upload from local. Click "Choose Files" and select all 4.
from google.colab import files
import shutil
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'{WORK}/{name}')
    shutil.copy(f'{WORK}/{name}', f'{DRIVE_DIR}/{name}')   # cache to Drive for next time
    print(f'  staged & backed up: {name}')

## Cell 5 — Load Qwen2.5-3B-Instruct in bf16 (clean, no double-allocation)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print('Pre-load mem (GB):', round(torch.cuda.memory_allocated() / 1e9, 2), '(should be ~0)')

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation='sdpa',
).to('cuda')

model.config.use_cache = False
print('Post-load mem (GB):', round(torch.cuda.memory_allocated() / 1e9, 2), '(should be ~6.18)')
print('If post-load > 8 GB, STOP and tell Claude — there is still a memory bug.')

## Cell 6 — Apply LoRA (r=16, lighter than v1)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.0,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model.enable_input_require_grads()
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print('Mem (GB) after LoRA:', round(torch.cuda.memory_allocated() / 1e9, 2))

## Cell 7 — Tokenize (MAX_LEN=384, no labels — collator handles it)

In [ ]:
import json
from datasets import Dataset

MAX_LEN = 384

def load_jsonl(path):
    return [json.loads(l) for l in open(path)]

train_raw = load_jsonl(f'{WORK}/dataset_v1_train.jsonl')
val_raw   = load_jsonl(f'{WORK}/dataset_v1_val.jsonl')
print(f'Raw: train={len(train_raw)}  val={len(val_raw)}')

def to_text(sample):
    return tok.apply_chat_template(sample['messages'], tokenize=False, add_generation_prompt=False)

def tokenize_fn(batch):
    texts = [to_text({'messages': m}) for m in batch['messages']]
    return tok(texts, truncation=True, max_length=MAX_LEN, padding=False)

train_ds = Dataset.from_list(train_raw).map(tokenize_fn, batched=True, batch_size=128, remove_columns=['messages'])
val_ds   = Dataset.from_list(val_raw  ).map(tokenize_fn, batched=True, batch_size=128, remove_columns=['messages'])

# Drop samples that hit the truncation cap (long verbatim quotes)
train_ds = train_ds.filter(lambda ex: len(ex['input_ids']) < MAX_LEN)
val_ds   = val_ds  .filter(lambda ex: len(ex['input_ids']) < MAX_LEN)
print(f'After length filter: train={len(train_ds)}  val={len(val_ds)}')
import numpy as np
lens = [len(x) for x in train_ds[:200]['input_ids']]
print(f'Length stats: min={min(lens)} median={int(np.median(lens))} max={max(lens)}')

## Cell 8 — Trainer (save every 150 steps, 2 epochs)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUT_DIR = f'{DRIVE_DIR}/checkpoints'   # save directly to Drive — survives disconnects
os.makedirs(OUT_DIR, exist_ok=True)

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,            # effective batch 16
    num_train_epochs=2,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=150,
    save_strategy='steps',
    save_steps=150,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=False,             # disabled for speed
    optim='adamw_torch',
    report_to='none',
    remove_unused_columns=False,
    label_names=['labels'],
    dataloader_num_workers=0,
)

collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    processing_class=tok,
)
n_steps_per_epoch = len(train_ds) // 16
print('Steps/epoch :', n_steps_per_epoch)
print('Total steps :', n_steps_per_epoch * 2)
print('Will save every 150 steps to:', OUT_DIR)

## Cell 9 — Callback: generate on val_prompts every 150 steps

In [ ]:
import re
from transformers import TrainerCallback

val_prompts = load_jsonl(f'{WORK}/val_prompts.jsonl')
SYSTEM_PROMPT = (
    'You are an expert on the Constitution of the Islamic Republic of Pakistan. '
    'Answer using only the text of the Constitution. Quote the relevant clause verbatim '
    'and end every answer with the citation in square brackets, e.g. [Art 25(1)] or '
    "[Art 260(1)]. If the question is not addressed in the Constitution, reply with: "
    "'This is not addressed in the Constitution of Pakistan. [no citation]'"
)
CITE_RE = re.compile(r'\[(Art|Part|Chapter|Preamble|no citation)')

@torch.inference_mode()
def generate_one(m, user_msg, max_new_tokens=256):
    msgs = [{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':user_msg}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt').to(m.device)
    m.eval()
    out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    m.train()
    return tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

class GenEvalCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, **kwargs):
        m = kwargs['model']
        n_cited = 0; total_chars = 0
        print(f'\n===== Step {state.global_step} — val_prompts =====')
        for p in val_prompts:
            ans = generate_one(m, p['prompt'])
            cited = bool(CITE_RE.search(ans))
            n_cited += int(cited); total_chars += len(ans)
            marker = '✓' if cited else '✗'
            print(f"  [{p['tier']}] {marker} Q{p['id']}: {ans[:120].replace(chr(10),' ')}...")
        print(f'  → citation rate: {n_cited}/{len(val_prompts)} · avg answer: {total_chars/len(val_prompts):.0f} chars')

trainer.add_callback(GenEvalCallback())
print('Callback registered.')

## Cell 10 — TRAIN (~75-90 min on T4)

If T4 disconnects: just re-run Cells 1, 3, 4, 5, 6, 7, 8, 9, then this cell with `resume_from_checkpoint=True`.

In [ ]:
result = trainer.train(resume_from_checkpoint=False)   # set True if resuming
print('\nTraining complete.')
print('Final loss:', result.training_loss)

## Cell 11 — Save LoRA + merge to FP16

In [ ]:
adapter_path = f'{DRIVE_DIR}/qwen25_3b_constitution_lora'
trainer.model.save_pretrained(adapter_path)
tok.save_pretrained(adapter_path)
print('[✓] LoRA adapter →', adapter_path)

del trainer
import gc; gc.collect(); torch.cuda.empty_cache()

from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation='sdpa',
).to('cuda')
merged = PeftModel.from_pretrained(base, adapter_path).merge_and_unload()

merged_path = f'{DRIVE_DIR}/qwen25_3b_constitution_v1'
merged.save_pretrained(merged_path, safe_serialization=True)
tok.save_pretrained(merged_path)
print('[✓] Merged FP16 model →', merged_path)

## Cell 12 — Final held-out eval

In [ ]:
held = load_jsonl(f'{WORK}/heldout_prompts.jsonl')
results = []; n_cited = 0; n_refuse = 0
print('===== Held-out eval (Q never seen) =====\n')
for p in held:
    ans = generate_one(merged, p['prompt'])
    cited = bool(re.search(r'\[(Art|Part|Chapter|Preamble)', ans))
    refused = '[no citation]' in ans
    n_cited += int(cited); n_refuse += int(refused)
    results.append({'id':p['id'],'tier':p['tier'],'q':p['prompt'],'a':ans,'cited':cited,'refused':refused})
    flag = '✓ CITED' if cited else ('REFUSED' if refused else '✗ NO-CITE')
    print(f"--- Q{p['id']} [{p['tier']}] {flag} ---")
    print(f"Q: {p['prompt']}")
    print(f"A: {ans[:500]}\n")

print(f'\n[summary] cited={n_cited}/10  refusals={n_refuse}/10  no-cite={10-n_cited-n_refuse}/10')

with open(f'{DRIVE_DIR}/heldout_eval_report.json','w') as f:
    json.dump({'results':results,'cited':n_cited,'refused':n_refuse}, f, indent=2)
print('[✓] Report →', f'{DRIVE_DIR}/heldout_eval_report.json')

## Cell 13 — Convert to GGUF Q4_K_M for llama-cpp-2 deployment

This is what EchoAI's local-LLM stack loads. Output: ~2 GB single file.

In [ ]:
import subprocess
# Clone llama.cpp if not already present
if not os.path.exists('/content/llama.cpp'):
    subprocess.run(['git','clone','--depth=1','https://github.com/ggerganov/llama.cpp.git','/content/llama.cpp'], check=True)
    subprocess.run(['pip','install','-q','-r','/content/llama.cpp/requirements.txt'], check=True)
print('llama.cpp ready.')

In [ ]:
# Step 1: HF → GGUF (FP16)
gguf_fp16 = f'{WORK}/qwen25_3b_constitution_v1.fp16.gguf'
subprocess.run([
    'python', '/content/llama.cpp/convert_hf_to_gguf.py',
    merged_path,
    '--outfile', gguf_fp16,
    '--outtype', 'f16',
], check=True)
print(f'[✓] FP16 GGUF: {gguf_fp16} ({os.path.getsize(gguf_fp16)/1e9:.2f} GB)')

In [ ]:
# Step 2: build llama-quantize then quantize FP16 → Q4_K_M
subprocess.run(['cmake','-S','/content/llama.cpp','-B','/content/llama.cpp/build','-DGGML_CUDA=OFF'], check=True)
subprocess.run(['cmake','--build','/content/llama.cpp/build','--target','llama-quantize','-j','--config','Release'], check=True)

gguf_q4 = f'{DRIVE_DIR}/qwen25_3b_constitution_v1.Q4_K_M.gguf'
subprocess.run([
    '/content/llama.cpp/build/bin/llama-quantize',
    gguf_fp16, gguf_q4, 'Q4_K_M',
], check=True)
print(f'[✓] Q4_K_M GGUF saved to Drive: {gguf_q4} ({os.path.getsize(gguf_q4)/1e9:.2f} GB)')
print('\nDeploy: copy this file into the EchoAI repo at models/ and update model resolver path.')

## Done. Artifacts in Drive `MyDrive/echoai_qwen_ft/`:
- `qwen25_3b_constitution_lora/` — LoRA adapter (~80 MB)
- `qwen25_3b_constitution_v1/` — Merged bf16 model (~6 GB)
- `qwen25_3b_constitution_v1.Q4_K_M.gguf` — Quantized deployment artifact (~2 GB) ← copy to EchoAI repo
- `heldout_eval_report.json` — Final eval scorecard
- `checkpoints/` — training checkpoints (can be deleted after Cell 11 succeeds)